# Theorem 19 — semantic-anchor necessity

**Formal source:** [`../19_semantic_anchor_necessity.md`](../19_semantic_anchor_necessity.md)

This Notebook is an executable finite witness, not the general proof. Passing it supports implementation consistency only; it does not establish learned-model or real-PHM evidence.

In [ ]:
import math
import itertools
import numpy as np
np.set_printoptions(precision=6, suppress=True)


def four_way(projectors, domain_index, atol=1e-9):
    ps = [np.asarray(p, float) for p in projectors]
    dimension = ps[0].shape[0]
    summed = sum(ps)
    values, vectors = np.linalg.eigh((summed + summed.T) / 2)
    basis = vectors[:, np.isclose(values, len(ps), atol=atol)]
    shared = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    basis = vectors[:, values > atol]
    union = basis @ basis.T if basis.size else np.zeros((dimension, dimension))
    observed = ps[domain_index]
    blocks = [shared, observed - shared, union - observed, np.eye(dimension) - union]
    for projector in blocks:
        np.testing.assert_allclose(projector, projector.T, atol=1e-8)
        np.testing.assert_allclose(projector @ projector, projector, atol=1e-8)
    for index, left in enumerate(blocks):
        for right in blocks[index + 1:]:
            np.testing.assert_allclose(left @ right, 0, atol=1e-8)
    np.testing.assert_allclose(sum(blocks), np.eye(dimension), atol=1e-8)
    return blocks


def normal_pdf(x, mean, standard_deviation):
    return np.exp(-0.5 * ((x - mean) / standard_deviation) ** 2) / (
        math.sqrt(2 * math.pi) * standard_deviation
    )

In [ ]:
rng = np.random.default_rng(19)
labels = np.tile([0, 1], 10000)
nuisance = rng.normal(size=labels.size)
representation = np.column_stack([labels, nuisance])
good = representation[:, 0].astype(int)
bad = 1 - labels
assert np.mean(good == labels) == 1
assert np.array_equal(np.sort(bad), np.sort(labels))
assert np.mean(bad == labels) == 0
print({"sufficient_anchor_accuracy": 1.0, "marginally_matched_reversal_accuracy": 0.0})

In [ ]:
print('THEORY_DEMO_PASS::19_semantic_anchor_necessity')
print('evidence_level: constructive_or_numerical_witness')
print('formal_claim_supported: false')